In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
from langchain_core.prompts import PromptTemplate

In [ ]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
splitted_data = splitter.split_documents(docs)

In [ ]:
len(splitted_data)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
vector_store = Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [ ]:
query = "Machine Learning & Data Learning"
data = vector_store.similarity_search(query=query)

In [ ]:
print(data[0])

In [ ]:
context = ""
for doc in data:
    context += doc.page_content + "\n"
print(context)

In [ ]:
res = llm.invoke(f"""Can you provide me answer based on provided context for my question, context={context} and question ={query}""")

### Chain - Context_Generated | Prompt | LLM | Strpraser

In [ ]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [ ]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistant and provide answered based on the context for user questions and if you don't know the answer then you can say 'I don't know'. Along with that generate 3 question that can be next follow-up questions that can be asked by the user on given context.
    Context:{context}
    Question:{question}
""")

In [ ]:
rag_chain = get_context | prompt | llm

In [ ]:
res = rag_chain.invoke("What's the RBC Count and is it good range?")

In [ ]:
print(res.content)